# Un PAM mínimo con software libre: la medición

Cuaderno de lectura de la medición `pam-minimo/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos): las pruebas de un laboratorio de gestión de accesos privilegiados montado con OpenBao 2.7.2 (bóveda), Warpgate 0.30.2 (bastión con TOTP, certificado SSH por sesión y grabación), un servidor SSH que solo admite certificados de la bóveda y PostgreSQL 17.11 con la tabla 28193 del INE como dato protegido. Los CSV los escribe `laboratorio.sh medir` (el medidor `medidor/medir.py` desde la red interna y `medidor/registros.py` desde el anfitrión). Este cuaderno solo los lee.

In [ ]:
import json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/pam-minimo/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")

## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## Las pruebas, por bloque

In [ ]:
pr = leer("pruebas.csv")
print(int(pr.ok.sum()), "de", len(pr), "pruebas con el resultado esperado")
print(pr.groupby("bloque", sort=True).ok.agg(["sum", "count"]).to_dict("index"))
pr[["bloque", "prueba", "esperado", "resultado", "ok", "ms"]]

## Certificados SSH y credenciales dinámicas

In [ ]:
c = leer("certificados.csv")
print(len(c), "certificado(s) ·", c.tipo.iloc[0], "· principales", c.principales.iloc[0], "· válido", int(c.valido_hasta.iloc[0] - c.valido_desde.iloc[0]), "s")
cr = leer("credenciales_bd.csv")
print(len(cr), "concesiones · TTL", sorted(cr.ttl_s.unique().tolist()), "s")
cr[["etiqueta", "ms_emision", "ttl_s"]]

## Lo que registró el servidor SSH

In [ ]:
s = leer("sshd_accesos.csv")
print(s.tipo.value_counts().sort_index().sort_values(ascending=False, kind="stable").to_dict())
print("key ID aceptados:", sorted(s[s.tipo == "acceso aceptado"].key_id.dropna().str.replace(r"warpgate:ana:[0-9a-f-]+", "warpgate:ana:<sesión>", regex=True).unique()))

## Grabaciones y sesiones del bastión

In [ ]:
g = leer("grabaciones.csv")
print(len(g), "grabaciones ·", int(g.octetos.sum()), "octetos ·", int(g.contiene_la_salida.sum()), "con la salida de la orden en claro")
ses = leer("sesiones.csv")
print(len(ses), "sesiones ·", int(ses.usuario.notna().sum()), "autenticadas ·", ses.protocolo.value_counts().sort_index().to_dict())

## Auditoría de la bóveda y del bastión

In [ ]:
a = leer("auditoria_openbao.csv")
print(int(a.n.sum()), "respuestas ·", int(a[a.error == "sí"].n.sum()), "con error")
print(a.groupby("quien").n.sum().sort_index().to_dict())
b = leer("auditoria_bastion.csv")
b.sort_values(["protocolo", "evento"])

## Qué cuesta

In [ ]:
e = leer("esfuerzo.csv"); cf = leer("configuracion.csv"); im = leer("imagenes.csv"); rc = leer("recursos.csv")
print("init:", int(e.segundos_acumulados.max()), "s ·", int(cf.lineas.sum()), "líneas en", len(cf), "ficheros")
print(dict(zip(im.imagen, im.tamano)))
print(dict(zip(rc.contenedor, rc.memoria)))
pr.plot.barh(x="prueba", y="ms", legend=False, figsize=(8, 9), logx=True, color="tab:blue"); plt.xlabel("ms (log)"); plt.tight_layout(); plt.show()